# Contrastive Learning, Semantic Search, and RAG

This notebook covers representation learning — training a mapping $f_\theta: x \mapsto \mathbb{R}^m$ such that semantically similar inputs land near each other in embedding space. We build up from first principles: cosine similarity, supervised pre-training, SimCLR's InfoNCE loss, hard negatives, brute-force semantic search, and a minimal RAG pipeline — all in plain NumPy.

## 1. Embeddings and Representation Learning

An **embedding** is the output of a mapping $f_\theta: x \mapsto \mathbb{R}^m$ applied to a raw input $x$ (image, text, audio, etc.). The goal is:

- **Similar inputs** $\Rightarrow$ **high cosine similarity** between their embeddings
- **Different inputs** $\Rightarrow$ **low cosine similarity**

**Cosine similarity** between two vectors $u, v \in \mathbb{R}^m$:

$$\text{sim}(u, v) = \frac{u \cdot v}{\|u\| \cdot \|v\|}$$

When embeddings are $\ell_2$-normalised ($\|u\| = \|v\| = 1$), cosine similarity reduces to the inner product $u^\top v$, which ranges from $-1$ (opposite) to $+1$ (identical direction).

Embeddings differ from supervised output heads: a classification head produces a probability distribution over a fixed label set, whereas an embedding is a dense continuous vector useful for any downstream similarity task — no label set needed at inference time.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

EMBEDDING_DIM = 8
NUM_SAMPLES   = 6   # 2 cats, 2 planes, 2 chairs

# Simulate three clusters of raw embeddings
cat_embeddings   = np.random.randn(2, EMBEDDING_DIM) + np.array([3, 0, 0, 0, 0, 0, 0, 0])
plane_embeddings = np.random.randn(2, EMBEDDING_DIM) + np.array([0, 3, 0, 0, 0, 0, 0, 0])
chair_embeddings = np.random.randn(2, EMBEDDING_DIM) + np.array([0, 0, 3, 0, 0, 0, 0, 0])

all_embeddings = np.vstack([cat_embeddings, plane_embeddings, chair_embeddings])
labels         = ["cat-1", "cat-2", "plane-1", "plane-2", "chair-1", "chair-2"]

def l2_normalize(X):
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    return X / (norms + 1e-8)

def cosine_similarity_matrix(X):
    X_norm = l2_normalize(X)
    return X_norm @ X_norm.T

sim_matrix = cosine_similarity_matrix(all_embeddings)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(sim_matrix, vmin=-1, vmax=1, cmap="RdYlGn")
ax.set_xticks(range(NUM_SAMPLES))
ax.set_yticks(range(NUM_SAMPLES))
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_yticklabels(labels)
plt.colorbar(im, ax=ax, label="cosine similarity")
ax.set_title("Cosine Similarity Matrix")
plt.tight_layout()
plt.show()

print("Intra-class (cat-1 vs cat-2):",  f"{sim_matrix[0,1]:.3f}")
print("Cross-class (cat-1 vs plane-1):", f"{sim_matrix[0,2]:.3f}")

## 2. Supervised Pre-training for Representations

Before contrastive methods, the standard approach was:

1. Train a deep classifier on a large labelled dataset (e.g. 1 000-class ImageNet).
2. The network has the form $y = W f_\theta(x)$, where $f_\theta(x) \in \mathbb{R}^m$ is the **penultimate layer** and $W \in \mathbb{R}^{K \times m}$ maps to $K$ class logits.
3. At transfer time, **discard $W$** and use $f_\theta(x)$ as the embedding.

Why does this work? To correctly classify $K$ semantically distinct categories with a *linear* head, the network must encode class-relevant structure in $f_\theta(x)$. Representations that carry no geometric structure cannot be separated by any linear hyperplane.

**Limitation:** embedding quality is bounded by label diversity. A binary label set (black/white image) forces $f_\theta$ to discard all colour information — the embeddings become useless for colour-based queries.

In [ ]:
np.random.seed(42)

# --- Tiny synthetic MLP classifier ---
# Input: 16-D feature, 2 hidden layers → 8-D penultimate → 3-class head

INPUT_DIM  = 16
HIDDEN_DIM = 32
EMBED_DIM  = 8
NUM_CLASSES = 3   # cat / plane / chair
N_PER_CLASS = 50

def relu(x):
    return np.maximum(0, x)

def softmax(x):
    e = np.exp(x - x.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

def cross_entropy(probs, y_one_hot):
    return -np.mean(np.sum(y_one_hot * np.log(probs + 1e-8), axis=1))

# Generate linearly separable synthetic data
class_means = np.array([[3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
                         [0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
                         [0,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0]], dtype=float)
X_data = np.vstack([np.random.randn(N_PER_CLASS, INPUT_DIM) + class_means[c]
                    for c in range(NUM_CLASSES)])
y_data = np.repeat(np.arange(NUM_CLASSES), N_PER_CLASS)
Y_oh   = np.eye(NUM_CLASSES)[y_data]

# Random initialisation
W1  = np.random.randn(INPUT_DIM, HIDDEN_DIM) * 0.1
W2  = np.random.randn(HIDDEN_DIM, EMBED_DIM) * 0.1
W3  = np.random.randn(EMBED_DIM, NUM_CLASSES) * 0.1

LR     = 0.05
EPOCHS = 300
losses = []

for epoch in range(EPOCHS):
    # Forward
    h1        = relu(X_data @ W1)
    embedding = relu(h1 @ W2)         # penultimate layer — this is f_θ(x)
    logits    = embedding @ W3
    probs     = softmax(logits)
    loss      = cross_entropy(probs, Y_oh)
    losses.append(loss)

    # Backward (manual)
    dlogits   = (probs - Y_oh) / len(X_data)
    dW3       = embedding.T @ dlogits
    dembed    = dlogits @ W3.T * (embedding > 0)
    dW2       = h1.T @ dembed
    dh1       = dembed @ W2.T * (h1 > 0)
    dW1       = X_data.T @ dh1

    W1 -= LR * dW1
    W2 -= LR * dW2
    W3 -= LR * dW3

# Extract penultimate embeddings
h1_final        = relu(X_data @ W1)
final_embeddings = relu(h1_final @ W2)

# PCA to 2D for visualisation
E_centred = final_embeddings - final_embeddings.mean(axis=0)
U, S, Vt  = np.linalg.svd(E_centred, full_matrices=False)
E_2d      = E_centred @ Vt[:2].T

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(losses)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Cross-entropy loss")
ax1.set_title("Training Loss")

colors = ["#e07b54", "#5b8db8", "#6bbf6b"]
class_names = ["cat", "plane", "chair"]
for c in range(NUM_CLASSES):
    idx = y_data == c
    ax2.scatter(E_2d[idx, 0], E_2d[idx, 1], label=class_names[c],
                color=colors[c], alpha=0.7, s=30)
ax2.set_title("Penultimate-layer Embeddings (PCA)")
ax2.legend()
plt.tight_layout()
plt.show()

preds = softmax(final_embeddings @ W3).argmax(axis=1)
print(f"Classification accuracy: {(preds == y_data).mean():.1%}")

## 3. Contrastive Learning Setup (SimCLR)

Contrastive learning learns embeddings **without any labels**. The key idea is to define similarity purely from data structure.

### Positive and negative pairs

Given an image $x_i$, apply two independent random augmentations:

$$\hat{x}_i = \text{aug}_1(x_i), \quad \tilde{x}_i = \text{aug}_2(x_i)$$

$(\hat{x}_i, \tilde{x}_i)$ is a **positive pair** — both views of the same underlying image.
Any $(\hat{x}_i, \tilde{x}_j)$ with $i \neq j$ is treated as a **negative pair**.

### Augmentation strategies

| Modality | Typical augmentations |
|---|---|
| Images | Random crop, horizontal flip, colour jitter, Gaussian blur, grayscale |
| Text | (title + headers) vs. full body of the same document |
| Image–text | Image $\hat{x}_i$ paired with a generated text description $\tilde{x}_i$ |

For text, the title of document A paired with the body of document A is a positive pair. The title of document A paired with the body of document B is a negative pair.

**Why augmentation works:** two crops of the same image share the same semantic content. The network must encode that shared content in order to bring the embeddings together — it cannot cheat by memorising pixel values.

In [ ]:
np.random.seed(42)

# Simulate augmentation on 1-D signals (stand-in for cropped image patches)
NUM_IMAGES  = 4
SIGNAL_LEN  = 64

def make_signal(freq, length):
    t = np.linspace(0, 2 * np.pi, length)
    return np.sin(freq * t)

def augment(signal, noise_scale=0.15, crop_fraction=0.75):
    """Random crop + Gaussian noise."""
    crop_len   = int(len(signal) * crop_fraction)
    start      = np.random.randint(0, len(signal) - crop_len)
    cropped    = signal[start : start + crop_len]
    # Pad back to original length with zeros (mirrors resize-after-crop)
    padded     = np.zeros(len(signal))
    padded[:crop_len] = cropped
    return padded + np.random.randn(len(signal)) * noise_scale

freqs = [1, 2, 4, 8]   # four distinct "images"
signals = [make_signal(f, SIGNAL_LEN) for f in freqs]

fig, axes = plt.subplots(NUM_IMAGES, 3, figsize=(12, 8))
for i, (sig, freq) in enumerate(zip(signals, freqs)):
    aug1 = augment(sig)
    aug2 = augment(sig)
    axes[i, 0].plot(sig,  color="#333")
    axes[i, 0].set_title(f"Original (freq={freq})", fontsize=9)
    axes[i, 1].plot(aug1, color="#5b8db8")
    axes[i, 1].set_title("Augmentation 1 = x̂", fontsize=9)
    axes[i, 2].plot(aug2, color="#e07b54")
    axes[i, 2].set_title("Augmentation 2 = x̃", fontsize=9)
    for ax in axes[i]:
        ax.set_yticks([])
        ax.set_xticks([])
plt.suptitle("Positive pairs: two augmented views of the same signal", fontsize=11)
plt.tight_layout()
plt.show()

## 4. InfoNCE Loss (SimCLR Loss)

Given a mini-batch of $B$ images, create $2B$ augmented views. Organise them as a $B \times B$ similarity matrix:

$$S_{ij} = \frac{f_\theta(\hat{x}_i)^\top f_\theta(\tilde{x}_j)}{\tau}$$

where $\tau > 0$ is a temperature hyper-parameter. The diagonal entries $S_{ii}$ are positive pairs; all off-diagonal entries are negative pairs.

The **InfoNCE loss** (also called NT-Xent in SimCLR) for one positive pair $(i, i)$ is:

$$\ell_i = -\log \frac{\exp(S_{ii})}{\sum_{j \neq i} \exp(S_{ij})}$$

Summed over all $i$:

$$\mathcal{L} = -\sum_{i=1}^{B} \log \frac{\exp(S_{ii})}{\sum_{j \neq i} \exp(S_{ij})}$$

This is exactly **softmax cross-entropy** where the "class label" for query $\hat{x}_i$ is the index of its matching positive view $\tilde{x}_i$. The denominator contains all $B-1$ negatives in the batch.

**Minimising $\mathcal{L}$** simultaneously:
- Maximises $S_{ii}$ (pulls positive pairs together)
- Minimises $S_{ij}$ for $j \neq i$ (pushes negative pairs apart)

In [ ]:
np.random.seed(42)

# --- InfoNCE from scratch on synthetic data ---
# 3 classes, 20 examples each, 16-D input → 8-D embedding (2-layer MLP)

N_CLASSES   = 3
N_PER_CLS   = 20
IN_DIM      = 16
EMBED_DIM   = 8
TEMPERATURE = 0.1
BATCH_SIZE  = 30   # 10 per class in each batch
LR          = 0.02
EPOCHS      = 400

class_centers = np.eye(N_CLASSES, IN_DIM) * 4.0
X_all = np.vstack([np.random.randn(N_PER_CLS, IN_DIM) + class_centers[c]
                   for c in range(N_CLASSES)])
y_all = np.repeat(np.arange(N_CLASSES), N_PER_CLS)

W1 = np.random.randn(IN_DIM,   EMBED_DIM * 2) * 0.05
W2 = np.random.randn(EMBED_DIM * 2, EMBED_DIM) * 0.05

def encode(X, W1, W2):
    H = relu(X @ W1)
    E = H @ W2
    return l2_normalize(E)

def augment_embedding(X, noise=0.3):
    return X + np.random.randn(*X.shape) * noise

def infonce_loss_and_grad(E_hat, E_tilde, temperature):
    """Returns scalar loss and gradient w.r.t. E_hat."""
    B   = len(E_hat)
    S   = (E_hat @ E_tilde.T) / temperature      # (B, B)
    # Mask diagonal as negative
    mask = np.eye(B, dtype=bool)
    # For each row i, label is column i
    S_shifted = S - S.max(axis=1, keepdims=True)  # numerical stability
    exp_S     = np.exp(S_shifted)
    exp_S_masked = exp_S * (~mask)                # zero out diagonal from denominator
    denom     = exp_S_masked.sum(axis=1, keepdims=True)
    loss      = -np.mean(S_shifted[mask] - np.log(denom.squeeze() + 1e-8))

    # Gradient of loss w.r.t. S, then chain to E_hat
    dS = exp_S_masked / (denom + 1e-8)            # (B, B)
    dS[mask] -= 1.0
    dS /= (B * temperature)
    dE_hat = dS @ E_tilde
    return loss, dE_hat

losses = []

for epoch in range(EPOCHS):
    idx   = np.random.choice(len(X_all), BATCH_SIZE, replace=False)
    X_bat = X_all[idx]

    X_hat   = augment_embedding(X_bat)
    X_tilde = augment_embedding(X_bat)

    # Forward through encoder
    H_hat   = relu(X_hat   @ W1)
    E_hat   = l2_normalize(H_hat   @ W2)
    H_tilde = relu(X_tilde @ W1)
    E_tilde = l2_normalize(H_tilde @ W2)

    loss, dE_hat = infonce_loss_and_grad(E_hat, E_tilde, TEMPERATURE)
    losses.append(loss)

    # Backprop through encoder (hat branch only for simplicity)
    dW2   = H_hat.T @ dE_hat
    dH    = (dE_hat @ W2.T) * (H_hat > 0)
    dW1   = X_hat.T @ dH

    W1 -= LR * dW1
    W2 -= LR * dW2

# Visualise final embeddings (all data)
E_final  = encode(X_all, W1, W2)
E_center = E_final - E_final.mean(axis=0)
_, _, Vt = np.linalg.svd(E_center, full_matrices=False)
E_2d     = E_center @ Vt[:2].T

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(losses)
ax1.set_xlabel("Step")
ax1.set_ylabel("InfoNCE loss")
ax1.set_title("InfoNCE Training (no labels used)")

colors = ["#e07b54", "#5b8db8", "#6bbf6b"]
for c in range(N_CLASSES):
    mask = y_all == c
    ax2.scatter(E_2d[mask, 0], E_2d[mask, 1],
                color=colors[c], label=f"class {c}", alpha=0.8, s=40)
ax2.set_title("Learned Embeddings (PCA) — no labels during training")
ax2.legend()
plt.tight_layout()
plt.show()

print(f"Final InfoNCE loss: {losses[-1]:.4f}")

## 5. Hard Negatives

Not all negatives are equal. Consider the InfoNCE denominator:

$$\sum_{j \neq i} \exp\!\left(\frac{S_{ij}}{\tau}\right)$$

If a negative pair $(i, j)$ produces a very low similarity score $S_{ij} \ll 0$, then $\exp(S_{ij}/\tau) \approx 0$. The loss gradient with respect to the encoder parameters contributed by that pair is negligible — the network has already solved that case and learns nothing further from it. These are **easy negatives**.

**Hard negatives** have $S_{ij}$ close to (or exceeding) $S_{ii}$, meaning the network currently confuses them with the positive. The gradient is large, providing a strong training signal.

### Strategies for obtaining hard negatives

1. **Larger batch size** — with $B$ images, there are $B-1$ negatives per query. More negatives stochastically includes more hard ones.
2. **Domain-coherent sampling** — sample all $B$ images from the same source (same programming language, same image category, same document corpus). Intra-domain pairs are naturally more confusable.
3. **Explicit negative mining** — filter negatives that exceed a similarity threshold before including them in the batch.

**Gradient analysis:** the gradient of $\mathcal{L}$ w.r.t. the encoder output for negative pair $(i,j)$ is proportional to:

$$p_{ij} = \frac{\exp(S_{ij}/\tau)}{\sum_{k \neq i} \exp(S_{ik}/\tau)}$$

Hard negatives have high $p_{ij}$; easy negatives have $p_{ij} \approx 0$.

In [ ]:
np.random.seed(42)

# Visualise gradient weight p_ij as a function of similarity score
tau  = 0.1
s_pos = 0.8   # positive pair similarity (fixed)
B     = 8     # batch size

# Vary the similarity of one negative while holding others at -0.2
s_neg_varied = np.linspace(-1.0, 0.9, 200)
other_negs   = np.full(B - 2, -0.2)   # remaining B-2 negatives

grad_weights = []
for s_hard in s_neg_varied:
    all_neg_sims = np.concatenate([[s_hard], other_negs])
    denom = np.sum(np.exp(all_neg_sims / tau))
    p_hard = np.exp(s_hard / tau) / denom
    grad_weights.append(p_hard)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(s_neg_varied, grad_weights, color="#5b8db8", lw=2)
ax.axvline(x=s_pos, color="#e07b54", linestyle="--", label=f"positive similarity = {s_pos}")
ax.set_xlabel("Negative pair similarity score $S_{ij}$")
ax.set_ylabel("Gradient weight $p_{ij}$")
ax.set_title("Hard negatives carry exponentially larger gradient weight")
ax.legend()
ax.annotate("easy negative\n(near-zero gradient)",
            xy=(-0.8, 0.01), fontsize=9, color="#888")
ax.annotate("hard negative\n(large gradient)",
            xy=(0.5, 0.6), fontsize=9, color="#e07b54")
plt.tight_layout()
plt.show()

print("Gradient weight at easy negative (sim=-0.8):",
      f"{np.interp(-0.8, s_neg_varied, grad_weights):.4f}")
print("Gradient weight at hard negative (sim=+0.7):",
      f"{np.interp(0.7,  s_neg_varied, grad_weights):.4f}")

## 6. Semantic Search with Embeddings

Semantic search uses the embedding model $f_\theta$ trained via contrastive learning (or supervised pre-training) as a retrieval engine.

### Two-phase architecture

**Offline (index time):**
Compute and store embeddings for every document in the corpus:

$$\mathcal{D} = \{f_\theta(d_1),\, f_\theta(d_2),\, \ldots,\, f_\theta(d_N)\}$$

**Online (query time):**
Given query $q$, embed it and find the nearest neighbour:

$$i^* = \arg\max_{i \in \{1,\ldots,N\}} \langle f_\theta(q),\, f_\theta(d_i) \rangle$$

### Complexity

| Method | Query time | Notes |
|---|---|---|
| Brute-force | $O(N \cdot m)$ | Exact; feasible for $N < 10^6$ |
| HNSW (Hierarchical NSW) | $O(\log N)$ | Approximate; graph-based |
| IVF (Inverted File Index) | $O(\sqrt{N})$ | Approximate; cluster-based |

**Vector databases** (FAISS, Pinecone, Weaviate, Chroma, Qdrant) implement these approximate nearest-neighbour algorithms and expose the index as a managed service. The embedding vectors are stored once; query latency is dominated by the $\arg\max$ operation, not by re-running the encoder.

In [ ]:
np.random.seed(42)

# --- Brute-force semantic search ---
# Synthetic "documents": each document is a cluster of 8-D embeddings.
# A query embedding is drawn near one cluster; we verify retrieval returns
# documents from the correct cluster.

N_DOCS      = 120
N_TOPICS    = 4     # distinct semantic topics
EMBED_DIM   = 8
N_QUERIES   = 6

topic_centers = np.random.randn(N_TOPICS, EMBED_DIM) * 3.0
doc_labels    = np.repeat(np.arange(N_TOPICS), N_DOCS // N_TOPICS)
doc_embeddings = np.vstack([
    np.random.randn(N_DOCS // N_TOPICS, EMBED_DIM) * 0.5 + topic_centers[t]
    for t in range(N_TOPICS)
])
doc_embeddings = l2_normalize(doc_embeddings)

# Sample queries: each drawn close to one topic
query_topics    = np.random.choice(N_TOPICS, N_QUERIES)
query_embeddings = l2_normalize(
    np.array([topic_centers[t] + np.random.randn(EMBED_DIM) * 0.3
              for t in query_topics])
)

def brute_force_top_k(query_emb, corpus_emb, k=5):
    scores = corpus_emb @ query_emb          # (N,)
    top_k  = np.argsort(scores)[::-1][:k]
    return top_k, scores[top_k]

print(f"{'Query':>6}  {'True topic':>10}  {'Top-5 retrieved topics':>30}  {'All correct?':>12}")
print("-" * 68)
all_correct = []
for q_idx, (q_emb, q_topic) in enumerate(zip(query_embeddings, query_topics)):
    top_k_idx, top_k_scores = brute_force_top_k(q_emb, doc_embeddings, k=5)
    retrieved_topics = doc_labels[top_k_idx]
    correct = all(retrieved_topics == q_topic)
    all_correct.append(correct)
    print(f"{q_idx:>6}  {q_topic:>10}  {str(list(retrieved_topics)):>30}  {'yes' if correct else 'no':>12}")

print(f"\nPrecision@5: {np.mean(all_correct):.1%}")

# Visualise with PCA
all_vecs  = np.vstack([doc_embeddings, query_embeddings])
centered  = all_vecs - all_vecs.mean(axis=0)
_, _, Vt  = np.linalg.svd(centered, full_matrices=False)
all_2d    = centered @ Vt[:2].T
doc_2d    = all_2d[:N_DOCS]
q_2d      = all_2d[N_DOCS:]

fig, ax = plt.subplots(figsize=(7, 6))
topic_colors = ["#e07b54", "#5b8db8", "#6bbf6b", "#b87fc9"]
for t in range(N_TOPICS):
    mask = doc_labels == t
    ax.scatter(doc_2d[mask, 0], doc_2d[mask, 1],
               color=topic_colors[t], alpha=0.4, s=20, label=f"topic {t} docs")
for i, (qv, qt) in enumerate(zip(q_2d, query_topics)):
    ax.scatter(qv[0], qv[1], color=topic_colors[qt], marker="*",
               s=200, edgecolors="black", linewidths=0.8)
ax.set_title("Semantic Search — stars = queries, dots = corpus\n(star colour = correct topic)")
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

## 7. Retrieval-Augmented Generation (RAG)

### Motivation

A pretrained LLM has fixed weights encoding knowledge from its training corpus. It cannot access:
- Private enterprise documents
- Information created after the training cutoff
- Sensitive personal data that must not leave a private environment

### Two approaches to inject private knowledge

| Approach | How it works | Drawbacks |
|---|---|---|
| Fine-tuning | Continue training on private data | Expensive; hard to delete specific facts later; requires hosting large model weights |
| **RAG** | At query time, retrieve relevant docs → inject into context | Cheap; easy deletion (remove from corpus); modular; access control is a retrieval filter |

### RAG pipeline

$$q \xrightarrow{\text{embed}} f_\theta(q) \xrightarrow{\text{retrieve}} \{d_{i_1}, \ldots, d_{i_k}\} \xrightarrow{\text{inject}} \text{LLM}([d_{i_1}, \ldots, d_{i_k}, q]) \rightarrow \text{answer}$$

### Data governance advantages of RAG

- **Access control:** filter retrieved documents by user permissions before injecting into context — the LLM never sees restricted content.
- **Deletion:** remove a document from the index → it is never retrieved again. No need to retrain or un-fine-tune the model (which is an open research problem).
- **Auditability:** every answer is traceable to the specific retrieved documents.

In [ ]:
np.random.seed(42)

# --- Minimal RAG pipeline (no external libraries) ---
# We use a bag-of-words TF-IDF-style embedding as a stand-in for a
# trained neural encoder. The retrieval and prompt-formatting logic
# is identical to what a real RAG system does.

CORPUS = [
    "The transformer architecture uses self-attention to model long-range dependencies.",
    "Convolutional neural networks are efficient for local feature extraction in images.",
    "Contrastive learning trains embeddings without labels using positive and negative pairs.",
    "The InfoNCE loss is equivalent to cross-entropy over augmented views in a batch.",
    "Retrieval-augmented generation injects private documents into the LLM context window.",
    "Vector databases store precomputed embeddings and support approximate nearest-neighbour search.",
    "Hard negatives are semantically similar but non-matching pairs that improve embedding quality.",
    "Fine-tuning on private data is expensive and makes information deletion difficult.",
    "SimCLR generates positive pairs by applying two random augmentations to the same image.",
    "Temperature tau in InfoNCE controls the sharpness of the similarity distribution.",
]

QUERIES = [
    "How does contrastive learning avoid needing labels?",
    "Why is RAG preferable to fine-tuning for private data?",
    "What is the InfoNCE loss?",
]

def tokenize(text):
    return set(text.lower().replace(".", "").replace(",", "").replace("?", "").split())

def build_vocab(texts):
    vocab = {}
    for text in texts:
        for token in tokenize(text):
            if token not in vocab:
                vocab[token] = len(vocab)
    return vocab

def bow_embed(text, vocab):
    vec = np.zeros(len(vocab))
    for token in tokenize(text):
        if token in vocab:
            vec[vocab[token]] += 1.0
    return l2_normalize(vec.reshape(1, -1)).squeeze()

def retrieve_top_k(query_emb, corpus_embs, corpus_texts, k=3):
    scores   = corpus_embs @ query_emb
    top_idx  = np.argsort(scores)[::-1][:k]
    return [(corpus_texts[i], float(scores[i])) for i in top_idx]

def format_rag_prompt(query, retrieved_docs):
    context_block = "\n".join(f"  [{i+1}] {doc}" for i, (doc, _) in enumerate(retrieved_docs))
    return (
        f"Context documents:\n{context_block}\n\n"
        f"Question: {query}\n"
        f"Answer (using only the context above):"
    )

# Build index
vocab        = build_vocab(CORPUS + QUERIES)
corpus_embs  = np.array([bow_embed(doc, vocab) for doc in CORPUS])

print(f"Corpus size: {len(CORPUS)} documents")
print(f"Vocabulary size: {len(vocab)} tokens")
print(f"Embedding dimension: {len(vocab)}  (bag-of-words)")
print("=" * 70)

for query in QUERIES:
    q_emb     = bow_embed(query, vocab)
    retrieved = retrieve_top_k(q_emb, corpus_embs, CORPUS, k=3)
    prompt    = format_rag_prompt(query, retrieved)
    print(f"\nQUERY: {query}")
    print("RETRIEVED DOCS (top-3):")
    for rank, (doc, score) in enumerate(retrieved, 1):
        print(f"  [{rank}] (score={score:.3f}) {doc}")
    print("\nFORMATTED PROMPT TO LLM:")
    print(prompt)
    print("-" * 70)